# 05 · Versiones, time travel y mantenimiento

⏱ **12 min** · Slides: 6–8 (el log y ACID) y 13–16 (versiones, time travel, OPTIMIZE / ZORDER / VACUUM, DeltaTable)

**Objetivo:** usar el historial que Delta fue creando mientras construíamos el pipeline.

| Paso | Qué haces |
|---|---|
| 1 | Leer el historial de silver |
| 2 | Consultar versiones anteriores (`VERSION AS OF`) |
| 3 | Un `DELETE` por error y `RESTORE` |
| 4 | `OPTIMIZE` + `ZORDER BY` |
| 5 | `VACUUM` y su precio |
| 6 | Lo mismo desde PySpark (`DeltaTable`) |

Los pasos 3 a 6 se hacen sobre una **copia** (`silver_demo`) para no alterar el pipeline.

**Requiere:** notebook 03.

In [0]:
%run ./00_Setup

## Paso 1 · El historial de silver

Cada `MERGE` del notebook 03 dejó una versión.

In [0]:
%sql
DESCRIBE HISTORY silver_episodes

In [0]:
%sql
DESCRIBE DETAIL silver_episodes

👀 **Qué observar:** `operation`, `operationMetrics` (filas insertadas / actualizadas / borradas) y, en `DETAIL`, el formato Delta y el número de archivos.

## Paso 2 · Consultar el pasado: `VERSION AS OF`

In [0]:
for v in range(table_version(SILVER) + 1):
    filas = spark.sql(f"SELECT count(*) FROM {SILVER} VERSION AS OF {v}").first()[0]
    print(f"versión {v}: {filas} filas")

In [0]:
# TIMESTAMP AS OF: la tabla tal como estaba en un momento dado (aquí, el de la versión 1)
momento = spark.sql(f"DESCRIBE HISTORY {SILVER}").where("version = 1").first()["timestamp"]
filas = spark.sql(f"SELECT count(*) FROM {SILVER} TIMESTAMP AS OF '{momento}'").first()[0]
print(f"En {momento} la tabla tenía {filas} filas")

👀 **Qué observar:** cada versión es una foto completa de la tabla. Consultarla no la modifica.

## Paso 3 · Un `DELETE` por error y `RESTORE`

Trabajamos sobre una copia.

In [0]:
%sql
CREATE OR REPLACE TABLE silver_demo AS SELECT * FROM silver_episodes

In [0]:
%sql
DELETE FROM silver_demo WHERE airdate = (SELECT max(airdate) FROM silver_demo)

In [0]:
%sql
SELECT (SELECT count(*) FROM silver_demo)              AS filas_ahora,
       (SELECT count(*) FROM silver_demo VERSION AS OF 0) AS filas_version_0

In [0]:
%sql
RESTORE TABLE silver_demo TO VERSION AS OF 0

In [0]:
%sql
DESCRIBE HISTORY silver_demo

👀 **Qué observar:** el `DELETE` creó la versión 1; `RESTORE` creó la 2 y devolvió las filas. El historial no se pierde:
el propio `RESTORE` queda auditado.

## Paso 4 · `OPTIMIZE` + `ZORDER BY`

Compacta archivos pequeños y agrupa las filas de una misma serie.

In [0]:
%sql
OPTIMIZE silver_demo ZORDER BY (show_id)

👀 **Qué observar:** `numFilesRemoved` / `numFilesAdded`. Con una tabla pequeña el efecto es mínimo; con miles de archivos es la diferencia entre lento y rápido.

## Paso 5 · `VACUUM` y su precio

Los archivos de las versiones viejas siguen en el almacenamiento hasta que `VACUUM` los borra.
Por defecto conserva 7 días; **solo para el laboratorio** bajamos la retención a 0.

In [0]:
%sql
VACUUM silver_demo DRY RUN

In [0]:
%sql
ALTER TABLE silver_demo SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')

In [0]:
%sql
VACUUM silver_demo RETAIN 0 HOURS

In [0]:
try:
    spark.sql("SELECT count(*) FROM silver_demo VERSION AS OF 0").first()
    print("La versión 0 todavía se puede leer")
except Exception as e:
    print("❌ La versión 0 ya no existe:", str(e).splitlines()[0][:160])

👀 **Qué observar:** el primer `DRY RUN` no borraría nada (todo tiene menos de 7 días). Tras bajar la retención,
`VACUUM` liberó espacio y el time travel a la versión 0 dejó de funcionar. **En producción: 7 días o más.**

## Paso 6 · Lo mismo desde PySpark: `DeltaTable`

In [0]:
dt = DeltaTable.forName(spark, qualified_name("silver_demo"))

display(dt.history().select("version", "operation", "operationMetrics"))   # DESCRIBE HISTORY
dt.optimize().executeZOrderBy("show_id")                                     # OPTIMIZE ZORDER BY
print("Versión actual:", dt.history().agg(F.max("version")).first()[0])

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_demo")

## ✔ Checkpoint

- [ ] Leí el historial y las métricas de cada operación
- [ ] Consulté versiones anteriores sin modificar la tabla
- [ ] Recuperé un borrado con `RESTORE`
- [ ] Sé qué hacen `OPTIMIZE`, `ZORDER BY` y `VACUUM`, y qué se pierde con `VACUUM`

**Siguiente:** `06_Unity_Catalog`